# Day 8 — Baseline Predictive Model: Logistic Regression

**Goal:** Train an interpretable baseline model to establish a performance floor. All future models (XGBoost, etc.) must beat this to justify added complexity.

## Sections
1. Setup & Data Load
2. Train/Test Split
3. Train Logistic Regression
4. Evaluate — The Big 5 Metrics
5. Confusion Matrix — Finding Hidden Losses
6. Credit Risk Metrics (Gini, KS)
7. Rule-Based Baseline Comparison
8. Coefficient Interpretation
9. Results Summary

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42

## 1. Setup & Data Load

In [ ]:
# Prefer eda-ready → validated → cleaned (whichever exists)
for fname in ['credit_default_eda_ready.parquet',
              'credit_default_validated.parquet',
              'credit_default_cleaned.parquet']:
    path = PROJECT_ROOT / 'data/processed' / fname
    if path.exists():
        df = pd.read_parquet(path)
        print(f'Loaded: {fname}  |  Shape: {df.shape}')
        break

df.head(3)

## 2. Train / Test Split

`stratify=y` ensures both splits preserve the ~22% default rate — essential for imbalanced data.

In [ ]:
# Drop any temp/derived columns that may have been saved from EDA notebooks
drop_cols = ['edu_label', 'mar_label', 'age_group']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

X = df.drop('default', axis=1)
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Train default rate: {y_train.mean():.3f}')
print(f'Test  default rate: {y_test.mean():.3f}  ← stratify is working')

## 3. Train Logistic Regression

Key choices:
- **StandardScaler** — LR is sensitive to feature scale; bill amounts (0–1M NTD) would dominate AGE (21–79) without scaling
- **`class_weight='balanced'`** — automatically upweights the minority class (defaulters) to address imbalance
- **`max_iter=1000`** — default 100 is too low for convergence on this feature set
- **Pipeline** — scaler + model in one object prevents data leakage during cross-validation

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

pipeline.fit(X_train, y_train)
print('Model trained.')

## 4. Evaluate — The Big 5 Metrics

> **Interview tip:** "Accuracy is 78% even for a model that never predicts default. That's why I report Precision, Recall, F1, and AUC-ROC — these are meaningful for imbalanced credit data."

In [ ]:
y_pred      = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

auc = roc_auc_score(y_test, y_pred_prob)
print(f'ROC-AUC: {auc:.4f}')

## 5. Confusion Matrix — Finding the Hidden Losses

**False Negatives** (bottom-left cell) = defaulters the model cleared as safe. These are "Hidden Losses" — each one costs ~97,500 NTD in expected loss (from Day 6 analysis).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Default', 'Default'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13)

# Normalized
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=['No Default', 'Default'])
disp_norm.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13)

plt.suptitle('Logistic Regression — Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/13_confusion_matrix_lr.png', dpi=150)
plt.show()

print(f'True  Positives (caught defaults):      {tp:,}')
print(f'False Negatives (missed defaults):      {fn:,}  ← Hidden Losses')
print(f'False Positives (wrongly flagged):      {fp:,}')
print(f'True  Negatives (correct approvals):    {tn:,}')
print(f'\nEstimated capital at risk from FN: {fn * 97_500:,.0f} NTD')

## 6. Credit Risk Metrics — Gini & KS

AUC-ROC is standard in ML. In credit risk, **Gini** and **KS statistic** are the industry-standard reporting metrics (Basel III, model validation teams).

- **Gini = 2 × AUC − 1** (ranges 0–1; higher is better)
- **KS statistic** = max separation between TPR and FPR curves across all thresholds

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

gini = 2 * auc - 1
ks   = max(tpr - fpr)
ks_threshold = thresholds[np.argmax(tpr - fpr)]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#2196F3', lw=2, label=f'Logistic Regression (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')

# Mark KS point
ks_idx = np.argmax(tpr - fpr)
ax.annotate(f'KS={ks:.3f}\n@ thresh={ks_threshold:.2f}',
            xy=(fpr[ks_idx], tpr[ks_idx]),
            xytext=(fpr[ks_idx] + 0.1, tpr[ks_idx] - 0.1),
            arrowprops=dict(arrowstyle='->', color='red'),
            color='red', fontsize=10)

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — Logistic Regression Baseline', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/14_roc_curve_lr.png', dpi=150)
plt.show()

print(f'AUC-ROC : {auc:.4f}')
print(f'Gini    : {gini:.4f}  (industry benchmark: >0.40 is acceptable, >0.60 is good)')
print(f'KS Stat : {ks:.4f}   (industry benchmark: >0.30 is acceptable, >0.45 is good)')

## 7. Rule-Based Baseline Comparison

From Day 5: `PAY_0 >= 2` is the single strongest predictor. How does LR compare to this naive rule?

In [ ]:
# Rule-based predictions on test set
y_rule = (X_test['PAY_0'] >= 2).astype(int)
rule_auc = roc_auc_score(y_test, X_test['PAY_0'])  # treat PAY_0 as a score

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

results = pd.DataFrame({
    'Model': ['Rule: PAY_0 >= 2', 'Logistic Regression'],
    'Accuracy':  [accuracy_score(y_test, y_rule),  accuracy_score(y_test, y_pred)],
    'Precision': [precision_score(y_test, y_rule), precision_score(y_test, y_pred)],
    'Recall':    [recall_score(y_test, y_rule),    recall_score(y_test, y_pred)],
    'F1':        [f1_score(y_test, y_rule),        f1_score(y_test, y_pred)],
    'AUC-ROC':   [rule_auc,                         auc],
    'Gini':      [2*rule_auc - 1,                   gini],
})

print(results.set_index('Model').round(4).to_string())

## 8. Coefficient Interpretation

LR coefficients (after scaling) show the direction and magnitude of each feature's contribution to default probability. This is the "explainability" that regulators expect for a baseline model.

In [ ]:
coefs = pd.Series(
    pipeline.named_steps['model'].coef_[0],
    index=X.columns
).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#F44336' if v > 0 else '#2196F3' for v in coefs.head(20)]
coefs.head(20).plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Top 20 LR Coefficients (scaled) — Red = increases default risk', fontsize=12)
ax.set_xlabel('Coefficient Value')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/15_lr_coefficients.png', dpi=150)
plt.show()

print('Top 10 risk-increasing features (positive coef):')
print(coefs[coefs > 0].head(10).to_string())
print('\nTop 5 risk-decreasing features (negative coef):')
print(coefs[coefs < 0].head(5).to_string())

## 9. Results Summary

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

print('=' * 55)
print('LOGISTIC REGRESSION BASELINE — FINAL RESULTS')
print('=' * 55)
print(f'  Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred):.4f}  ← priority metric')
print(f'  F1 Score  : {f1_score(y_test, y_pred):.4f}')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  Gini      : {gini:.4f}')
print(f'  KS Stat   : {ks:.4f}')
print(f'  Hidden Losses (FN): {fn:,} customers  ~{fn * 97_500:,.0f} NTD at risk')
print('=' * 55)
print('Next: Day 9 — XGBoost must beat these numbers.')